In [ ]:
#@title 🎨 InYogeshwar AI Voice Cloner
from IPython.display import HTML

HTML("""
<div style="background:#0f172a;padding:25px;border-radius:18px;color:white;text-align:center;">
  <img src="https://i.ibb.co/ycyZ3yG0/banner.png"
       style="width:100%;max-width:1000px;border-radius:15px;box-shadow:0 0 25px rgba(255,77,0,0.6);">
  <h1 style="color:#ff4d00;margin-top:18px;">🔥 InYogeshwar AI Voice Cloner</h1>
  <p style="font-size:20px;">Clone Any Voice in Seconds 🚀</p>
  <p>
    🌐 <a href="https://www.inyogeshwar.co.in" style="color:#facc15;">Website</a> |
    📺 <a href="https://www.youtube.com/@in_yogeshwar" style="color:#facc15;">YouTube</a> |
    📸 <a href="https://www.instagram.com/in_yogeshwar/" style="color:#facc15;">Instagram</a>
  </p>
  <p style="color:orange;">⚠️ Use responsibly. Only clone voices you have permission for.</p>
</div>
""")

In [1]:
#@title 🚀 One‑Click Install & Cleanup
import os, shutil, zipfile, subprocess

print("🧹 Cleaning previous RVC installation...")
for folder in ['/content/RVC', '/content/dataset', '/content/sample_data', '/content/Index_Temp']:
    if os.path.exists(folder):
        shutil.rmtree(folder, ignore_errors=True)
        print(f"  Removed {folder}")

print("📥 Downloading RVC...")
!wget -q --show-progress -O /content/RVC.zip https://huggingface.co/datasets/BahaaMahmoud88/testrvc/resolve/main/RVC.zip

if not os.path.exists('/content/RVC.zip') or os.path.getsize('/content/RVC.zip') < 5_000_000:
    raise FileNotFoundError("❌ RVC.zip is missing or corrupted!")

print("📦 Extracting RVC...")
with zipfile.ZipFile('/content/RVC.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/RVC')
os.remove('/content/RVC.zip')

print("⚙️ Installing dependencies (takes ~5 min)...")
!pip install uv > /dev/null 2>&1
!uv pip install --system torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 > /dev/null 2>&1
!uv pip install --system git+https://github.com/maxrmorrison/torchcrepe.git > /dev/null 2>&1

%cd /content/RVC
!uv pip install --system -r requirements-py312.txt > /dev/null 2>&1

# Patch fairseq
import fairseq
!sed -i 's/state = torch.load(f, map_location=torch.device("cpu"))/state = torch.load(f, map_location=torch.device("cpu"), weights_only=False)/' $(python -c "import fairseq; import os; print(os.path.join(os.path.dirname(fairseq.__file__), 'checkpoint_utils.py'))")

print("\n✅ All done! Ready to train.")

🧹 Cleaning previous RVC installation...
  Removed /content/sample_data
📥 Downloading RVC...
/content/RVC.zip    100%[===================>]   1.72G   270MB/s    in 7.9s    
📦 Extracting RVC...
⚙️ Installing dependencies (takes ~5 min)...
/content/RVC


/usr/local/lib/python3.12/dist-packages/fairseq/models/wav2vec/wav2vec2_asr.py:553: SyntaxWarning: invalid escape sequence '\d'
  r = re.compile("encoder.layers.\d.")


2026-03-22 18:26:56.727017: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774204016.749194    6471 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774204016.756211    6471 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774204016.774381    6471 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774204016.774416    6471 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774204016.774419    6471 computation_placer.cc:177] computation placer alr

In [ ]:
#@title 📂 Upload Dataset (clean vocals)
#@markdown Upload one or more WAV/MP3 files (recommended: 3–10 min total, mono, 44.1 kHz).
#@markdown For best results, use a single long file (up to 30 min) – it will be split automatically.

import os, librosa, soundfile as sf
from google.colab import files
from IPython.display import Audio, display

os.makedirs('/content/dataset', exist_ok=True)
for f in os.listdir('/content/dataset'):
    os.remove(os.path.join('/content/dataset', f))

uploaded = files.upload()
if not uploaded:
    raise ValueError("❌ No file uploaded.")

for fn in uploaded.keys():
    print(f"Processing: {fn}")
    audio, sr = librosa.load(fn, sr=32000, mono=True)
    out_name = os.path.splitext(fn)[0] + '.wav'
    sf.write(f'/content/dataset/{out_name}', audio, 32000, format='wav')
    print(f"  → saved as {out_name}")

first_wav = next(f for f in os.listdir('/content/dataset') if f.endswith('.wav'))
display(Audio(f'/content/dataset/{first_wav}'))
print(f"✅ Dataset ready: {len(os.listdir('/content/dataset'))} WAV files.")

In [ ]:
#@title ⚙️ Preprocess & Build Index
import os, subprocess, shutil, numpy as np, faiss, traceback
from sklearn.cluster import MiniBatchKMeans
from IPython.display import clear_output, display
from ipywidgets import Button

%cd /content/RVC

model_name = ""  # @param {type:"string"}

# Create logs directory if it doesn't exist
logs_dir = f'/content/RVC/logs/{model_name}'
os.makedirs(logs_dir, exist_ok=True)
print(f"✅ Created logs directory: {logs_dir}")

# Resume support
temp_DG_path = '/content/temp_DG'
if os.path.exists(logs_dir):
    print("⏩ Existing model found – resuming training.")
    os.makedirs(temp_DG_path, exist_ok=True)
    for item in os.listdir(logs_dir):
        item_path = os.path.join(logs_dir, item)
        if os.path.isfile(item_path) and (item.startswith('D_') or item.startswith('G_')) and item.endswith('.pth'):
            shutil.copy(item_path, temp_DG_path)
    for item in os.listdir(logs_dir):
        item_path = os.path.join(logs_dir, item)
        if os.path.isfile(item_path):
            os.remove(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
    for f in os.listdir(temp_DG_path):
        shutil.move(os.path.join(temp_DG_path, f), logs_dir)
    shutil.rmtree(temp_DG_path)
else:
    print("🆕 New model.")

# Preprocess
print("\n1️⃣ Preprocessing audio...")
dataset_folder = '/content/dataset'
cmd = f'python infer/modules/train/preprocess.py {dataset_folder} 32000 2 ./logs/{model_name} False 3.0'
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("❌ Preprocessing failed:", result.stderr)
    raise RuntimeError("Preprocessing error. Check your audio files (must be WAV, mono, 32k).")
print("✅ Preprocessing done.")

# Check that 0_gt_wavs exists and has files
gt_dir = f'/content/RVC/logs/{model_name}/0_gt_wavs'
if not os.path.exists(gt_dir) or len(os.listdir(gt_dir)) == 0:
    raise FileNotFoundError(f"❌ No training data found in {gt_dir}. Preprocessing likely failed.")

# Extract F0
print("\n2️⃣ Extracting F0 (RMVPE)...")
cmd = f'python infer/modules/train/extract/extract_f0_rmvpe.py 1 0 0 ./logs/{model_name} True'
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("❌ F0 extraction failed:", result.stderr)
    raise RuntimeError("F0 extraction error.")
else:
    print("✅ F0 done.")

# Extract features
print("\n3️⃣ Extracting features...")
cmd = f'python infer/modules/train/extract_feature_print.py cuda:0 1 0 ./logs/{model_name} v2 True'
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("❌ Feature extraction failed:", result.stderr)
    raise RuntimeError("Feature extraction error.")
else:
    print("✅ Features done.")

# Build index
print("\n4️⃣ Building FAISS index...")
exp_dir = f'logs/{model_name}'
feature_dir = f'{exp_dir}/3_feature768'
if not os.path.exists(feature_dir) or len(os.listdir(feature_dir)) == 0:
    raise FileNotFoundError("No feature files found. Cannot build index.")

npys = [np.load(os.path.join(feature_dir, name)) for name in sorted(os.listdir(feature_dir))]
big_npy = np.concatenate(npys, 0)
np.random.shuffle(big_npy)

if big_npy.shape[0] > 200000:
    print(f"Large dataset ({big_npy.shape[0]} samples), running k-means reduction...")
    kmeans = MiniBatchKMeans(n_clusters=10000, batch_size=256*os.cpu_count(), verbose=True, init='random')
    big_npy = kmeans.fit(big_npy).cluster_centers_
    print(f"Reduced to {big_npy.shape[0]} cluster centers.")

np.save(f"{exp_dir}/total_fea.npy", big_npy)

n_ivf = min(int(16 * np.sqrt(big_npy.shape[0])), big_npy.shape[0] // 39)
index = faiss.index_factory(768, f"IVF{n_ivf},Flat")
index_ivf = faiss.extract_index_ivf(index)
index_ivf.nprobe = 1
index.train(big_npy)
faiss.write_index(index, f"{exp_dir}/trained_IVF{n_ivf}_Flat_nprobe_{model_name}_v2.index")
print("   Trained index.")

for i in range(0, big_npy.shape[0], 8192):
    index.add(big_npy[i:i+8192])
faiss.write_index(index, f"{exp_dir}/added_IVF{n_ivf}_Flat_nprobe_{model_name}_v2.index")
print(f"✅ Index built: added_IVF{n_ivf}_Flat_nprobe_{model_name}_v2.index")

clear_output()
display(Button(description="✔ Index Ready", button_style="success"))

In [ ]:
#@title 🎯 Train Model (set epochs, etc.)
%cd /content/RVC
import os, json, subprocess
from random import shuffle

model_name = ""  # @param {type:"string"}
epochs = 200        # @param {type:"slider", min:50, max:5000, step:10}
save_frequency = 50 # @param {type:"slider", min:10, max:100, step:10}
batch_size = 8      # @param {type:"slider", min:1, max:20, step:1}
sample_rate = '32k'

G_file = f'assets/pretrained_v2/f0Ov2Super{sample_rate}G.pth'
D_file = f'assets/pretrained_v2/f0Ov2Super{sample_rate}D.pth'

def train_model(exp_dir1, sr2, if_f0_3, spk_id5, save_epoch10, total_epoch11,
                batch_size12, if_save_latest13, pretrained_G14, pretrained_D15,
                gpus16, if_cache_gpu17, if_save_every_weights18, version19):

    exp_dir = f'/content/RVC/logs/{exp_dir1}'
    gt_wavs_dir = f"{exp_dir}/0_gt_wavs"
    feature_dir = f"{exp_dir}/3_feature768" if version19 == "v2" else f"{exp_dir}/3_feature256"
    if if_f0_3:
        f0_dir = f"{exp_dir}/2a_f0"
        f0nsf_dir = f"{exp_dir}/2b-f0nsf"
        names = (set([name.split(".")[0] for name in os.listdir(gt_wavs_dir)]) &
                 set([name.split(".")[0] for name in os.listdir(feature_dir)]) &
                 set([name.split(".")[0] for name in os.listdir(f0_dir)]) &
                 set([name.split(".")[0] for name in os.listdir(f0nsf_dir)]))
    else:
        names = set([name.split(".")[0] for name in os.listdir(gt_wavs_dir)]) & set([name.split(".")[0] for name in os.listdir(feature_dir)])

    if len(names) == 0:
        raise RuntimeError("No matching files found. Check preprocessing.")

    opt = []
    for name in names:
        if if_f0_3:
            opt.append(f"{gt_wavs_dir}/{name}.wav|{feature_dir}/{name}.npy|{f0_dir}/{name}.wav.npy|{f0nsf_dir}/{name}.wav.npy|{spk_id5}")
        else:
            opt.append(f"{gt_wavs_dir}/{name}.wav|{feature_dir}/{name}.npy|{spk_id5}")

    fea_dim = 768 if version19 == "v2" else 256
    for _ in range(2):
        if if_f0_3:
            opt.append(f"/content/RVC/logs/mute/0_gt_wavs/mute{sr2}.wav|/content/RVC/logs/mute/3_feature{fea_dim}/mute.npy|/content/RVC/logs/mute/2a_f0/mute.wav.npy|/content/RVC/logs/mute/2b-f0nsf/mute.wav.npy|{spk_id5}")
        else:
            opt.append(f"/content/RVC/logs/mute/0_gt_wavs/mute{sr2}.wav|/content/RVC/logs/mute/3_feature{fea_dim}/mute.npy|{spk_id5}")

    shuffle(opt)
    with open(f"{exp_dir}/filelist.txt", "w") as f:
        f.write("\n".join(opt))

    config_path = f"configs/v2/{sr2}.json" if version19 == "v2" else f"configs/v1/{sr2}.json"
    config_save = os.path.join(exp_dir, "config.json")
    if not os.path.exists(config_save):
        with open(config_save, "w") as f:
            with open(config_path) as cfg:
                json.dump(json.load(cfg), f, indent=4)

    cmd = (f'python infer/modules/train/train.py -e "{exp_dir1}" -sr {sr2} -f0 {1 if if_f0_3 else 0} '
           f'-bs {batch_size12} -g {gpus16} -te {total_epoch11} -se {save_epoch10} '
           f'{"-pg " + pretrained_G14 if pretrained_G14 else ""} '
           f'{"-pd " + pretrained_D15 if pretrained_D15 else ""} '
           f'-l {1 if if_save_latest13 else 0} -c {1 if if_cache_gpu17 else 0} '
           f'-sw {1 if if_save_every_weights18 else 0} -v {version19}')
    print("▶️ Training command:")
    print(cmd)
    print("\n📊 Training logs (may take a while)...\n")

    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    process.wait()
    if process.returncode != 0:
        raise RuntimeError("Training failed.")
    return "✅ Training finished."

cache = False  # Set to True if >16GB GPU memory
try:
    train_model(model_name, sample_rate, True, 0, save_frequency, epochs,
                batch_size, True, G_file, D_file, 0, cache, True, 'v2')
except Exception as e:
    print(f"❌ Training error: {e}")
else:
    print("\n🎉 Model training complete! You can now use it for inference.")

In [ ]:
#@title 💾 Backup Model to Google Drive
import os, tarfile, time
from google.colab import drive

def mount_drive(retries=3, delay=5):
    for attempt in range(retries):
        try:
            if not os.path.exists('/content/drive/MyDrive'):
                print(f"🔄 Mounting Drive (attempt {attempt+1}/{retries})...")
                drive.mount('/content/drive', force_remount=True)
                print("✅ Drive mounted successfully.")
                return
            else:
                print("✅ Drive already mounted.")
                return
        except Exception as e:
            print(f"❌ Mount failed: {e}")
            if attempt < retries - 1:
                print(f"⏳ Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                print("❌ Could not mount Drive after multiple attempts.")
                raise

mount_drive()

model_name = ""  # @param {type:"string"}
backup_dir = '/content/drive/MyDrive/RVC_Packages'
os.makedirs(backup_dir, exist_ok=True)

print(f"📦 Creating backup of {model_name}...")
with tarfile.open(f'{backup_dir}/{model_name}.tar.gz', 'w:gz') as tar:
    tar.add(f'/content/RVC/logs/{model_name}', arcname=f'logs/{model_name}')
    if os.path.exists(f'/content/RVC/assets/weights/{model_name}.pth'):
        tar.add(f'/content/RVC/assets/weights/{model_name}.pth', arcname=f'assets/weights/{model_name}.pth')
print(f"✅ Model backed up to {backup_dir}/{model_name}.tar.gz")

In [ ]:
#@title 📥 Load Model from Google Drive
import os, tarfile, time
from google.colab import drive

def mount_drive(retries=3, delay=5):
    for attempt in range(retries):
        try:
            if not os.path.exists('/content/drive/MyDrive'):
                print(f"🔄 Mounting Drive (attempt {attempt+1}/{retries})...")
                drive.mount('/content/drive', force_remount=True)
                print("✅ Drive mounted successfully.")
                return
            else:
                print("✅ Drive already mounted.")
                return
        except Exception as e:
            print(f"❌ Mount failed: {e}")
            if attempt < retries - 1:
                print(f"⏳ Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                print("❌ Could not mount Drive after multiple attempts.")
                raise

mount_drive()

model_name = ""  # @param {type:"string"}
backup_file = f'/content/drive/MyDrive/RVC_Packages/{model_name}.tar.gz'
if not os.path.exists(backup_file):
    raise FileNotFoundError(f"❌ Backup not found: {backup_file}")

local_copy = f'/content/{model_name}.tar.gz'
print("📥 Copying model file...")
!cp "{backup_file}" "{local_copy}"

print("📦 Extracting model...")
with tarfile.open(local_copy, 'r:gz') as tar:
    tar.extractall('/content/RVC')
os.remove(local_copy)
print(f"✅ Model {model_name} loaded successfully.")

In [ ]:
#@title 🎙️ Step 1: Upload Target Audio
import os, librosa, soundfile as sf
from google.colab import files
from IPython.display import Audio, display

os.makedirs('/content/sample_data', exist_ok=True)
target_path = '/content/sample_data/input_audio.wav'
if os.path.exists(target_path):
    os.remove(target_path)

print("📤 Upload audio to convert (WAV/MP3):")
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file uploaded.")

fn = list(uploaded.keys())[0]
audio, sr = librosa.load(fn, sr=32000, mono=True)
sf.write(target_path, audio, 32000)
display(Audio(target_path))
print("✅ Ready for conversion.")

In [ ]:
#@title 🎙️ Step 2: Run Conversion
%cd /content/RVC
import os, subprocess
from IPython.display import Audio

model_name = ""  # @param {type:"string"}
pitch = 0          # @param {type:"slider", min:-12, max:12, step:1}
index_rate = 0.5   # @param {type:"slider", min:0, max:1, step:0.01}
protect = 0.5      # @param {type:"slider", min:0, max:1, step:0.01}

model_path = f"/content/RVC/assets/weights/{model_name}.pth"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model {model_name}.pth not found. Did you train or load it?")

logs_dir = f"/content/RVC/logs/{model_name}"
index_file = None
for f in os.listdir(logs_dir):
    if f.startswith('added') and f.endswith('.index'):
        index_file = f
        break
if index_file is None:
    raise FileNotFoundError("No index file found.")
index_path = os.path.join(logs_dir, index_file)

input_path = "/content/sample_data/input_audio.wav"
output_path = "/content/RVC/audios/output_audio.wav"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

cmd = (
    f'python tools/cmd/infer_cli.py '
    f'--f0up_key {pitch} '
    f'--input_path "{input_path}" '
    f'--index_path "{index_path}" '
    f'--f0method "rmvpe" '
    f'--opt_path "{output_path}" '
    f'--model_name "{model_name}.pth" '
    f'--index_rate {index_rate} '
    f'--device "cuda:0" '
    f'--is_half True '
    f'--filter_radius 3 '
    f'--resample_sr 0 '
    f'--rms_mix_rate 0 '
    f'--protect {protect}'
)
subprocess.run(cmd, shell=True, check=True)
print("\n✅ Conversion done!")
Audio(output_path)

In [ ]:
#@title 📥 Download Output Audio
from google.colab import files
files.download('/content/RVC/audios/output_audio.wav')